
<center><h1>🐤 <b><u>Tweet Classification with MLP-Mixers ( TF-Keras )</u></b></h1></center>

![mlp_mixer_diagram](https://github.com/shubham0204/Google_Colab_Notebooks/blob/main/data/mlp_mixer_diagram.PNG?raw=true)

> Image Source: [MLP-Mixer: An all-MLP Architecture for Vision](https://arxiv.org/abs/2105.01601)


Entering the **"Natural Language Processing with Disaster Tweets"**, our goal is to design a ML model which can efficiently classify tweets into two categories, the ones which refer ( or describe ) to a 😱 disaster and others which don't contain any such context 😁.

> In this notebook, you'll learn how to implement an MLP-Mixer architecture for classifying with TensorFlow Keras.

Here's the list of all things we'll do ( and enjoy! ) in this notebook,

* First, we process the raw texts in order to 🧹 eliminate unwanted characters and symbols.
* We implement our MLP-Mixer model with TensorFlow and train it on the processed data.
* Evaluate the model and produce the `submission.csv` file.



## 1. 🛠 Processing the text data

### a. ✂️ **Reading and truncating the CSV data**

The data provided to us is in the CSV format. First, we parse the CSV file using `pandas.read_csv` which transform it into a `DataFrame` object, which eases data handling.


In [ ]:

# Importing the required packages.
from nltk.corpus import stopwords
import tensorflow as tf
import pandas as pd
import numpy as np
import sklearn
import re



As seen in the output of the code cell below, the CSV data contains 5 columns. Considering our problem, we'll truncate `df` and use `text` and `target` columns. The reason behind eliminating columns `keyword` and `location` is,

* As we classifying tweets, the `location` 🏠 of a tweet is insignificant. The column also contains several null entries ( interpreted as `NaN` ).
* The `keyword` column could be used as a *metadata* to the existing tweet. But we drop its use as it contains several `NaN` values. Even if we try to drop these rows containing `NaN` using `pandas.dropNa()`, we'll be left with insufficient data for training our model. ( Neural networks require larger amounts of data for training )

Also, we store the entries of these columns in NumPy arrays.


In [ ]:

# Read the CSV file using Pandas
df = pd.read_csv( '../input/nlp-getting-started/train.csv' )
df.head()


In [ ]:

# Truncate the dataFrame and use only `text` and `target` columns.
df = pd.read_csv( '../input/nlp-getting-started/train.csv' , usecols=[ 'text' , 'target' ] )
print( df.head() )

# Store the entries in the above mentioned columns in NumPy arrays.
raw_texts = df[ 'text' ].values
raw_labels = df[ 'target' ].values



In the output of the code cell below, you'll observe that we have unequal samples belonging to both the classes. This may affect our model's performance on any of the classes. Hence, we compute the class weights using `sklearn.utils.class_weight.compute_class_weight` which will be used for weighing the losses for each of the classes.
    

In [ ]:

# Print the no. of samples for each class.
print( df[ 'target' ].value_counts() )

# Compute the class weights
# See this answer -> https://datascience.stackexchange.com/a/69302/68023
class_weights = dict( zip( np.unique( raw_labels ), sklearn.utils.class_weight.compute_class_weight( 'balanced', np.unique( raw_labels ), raw_labels ))) 



### b. 🧹 **Cleaning and tokenizing the textual data**

![tokenization](https://www.kdnuggets.com/wp-content/uploads/text-tokens-tokenization-manning.jpg)

> Image Source: [Tokenization and Text Data Preparation - KDNuggets](https://www.kdnuggets.com/2020/03/tensorflow-keras-tokenization-text-data-prep.html)

As you might have observed, the text ( of the tweets ) contains hashtags and URLs to images attached with the tweet. These are unnecessary features and hence we use regular expressions to filter each tweet. We perform a two-step filtration process:

1. Convert the words in the tweet to lowercase. Also, remove all non-alphabet characters from the text, like numbers, punctuations. Refer to this [StackOverflow answer](https://stackoverflow.com/questions/22520932/python-remove-all-non-alphabet-chars-from-string).

2. Remove hyperlinks from the tweet. Refer to this [StackOverflow answer](https://stackoverflow.com/questions/11331982/how-to-remove-any-url-within-a-string-in-python/11332580)

3. Remove 😀 emojis from the tweet. Refer to this [StackOverflow answer](https://stackoverflow.com/a/33417311/10878733)

Finally, we tokenize all the sentences and store these tokens in `processed_tokens`. After eliminating all duplicate tokens, we assign an index to each token and create two dictionaries which map each token to its index and vice-versa. See `word_to_index` and `index_to_word`.



In [ ]:

# Regex to remove non-alphabet characters\
r1 = re.compile( '[^a-zA-Z ]' )

# Regex to remove hyperlinks ( "https://..." )
r2 = re.compile( 'http://\S+|https://\S+' )

# Regex to extract words starting with #
r3 = re.compile( '#(\w+)' )

# Remove non-alphabet char from given sentence. See https://stackoverflow.com/questions/22520932/python-remove-all-non-alphabet-chars-from-string
def remove_non_alphabet_char( sent ):
    return r1.sub( '' , sent )

# Remove hyperlinks from given sentence. See https://stackoverflow.com/questions/11331982/how-to-remove-any-url-within-a-string-in-python/11332580
def remove_links( sent ):
    return r2.sub( '' , sent )

# Remove emojis. See https://stackoverflow.com/a/33417311/10878733
def remove_emojis( sent ):
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF" 
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub( r'',sent )
    

# Clean the given sentence ( using the two methods above ) and tokenize it.
# Also, remove stop words from the sentence.
def process_sent( sent ):
    sent = text.lower()
    sent = remove_links( sent )
    sent = remove_non_alphabet_char( sent )
    sent = remove_emojis( sent )
    tokens = sent.split()
    tokens = [ token.strip() for token in tokens if token not in stopwords.words( 'english' ) ]
    return tokens
    
# Collect tokens and tokenized sentences in two arrays.
processed_tokens = []
tokenized_sentences = []
for text in raw_texts:
    tokens = process_sent( text )
    processed_tokens += tokens
    tokenized_sentences.append( tokens )

# Get unique tokens
unique_tokens = list( set( processed_tokens ) )
unique_tokens = np.array( unique_tokens )

# Compute vocabulary size ( will be used for the Embedding layer )
vocab_size = len( unique_tokens )
# Create an array of indexed starting from 1 to vocab_size + 1 
# For ex. [ 1 , 2 , 3 , ... , vocab_size ]
indices = np.arange( 1 , vocab_size + 1 )

# Zip unique_tokens and indices to create a dict with elements ( index , token ) where index has dtype=int and
# token has dtype=str
# This dict maps every index to its corresponding token.
int_to_word = dict( zip( indices , unique_tokens ) )

# This dict maps every token to its corresponding index.
word_to_int = dict( zip( unique_tokens , indices ) )



Our next step is to transform each tokenized sentence into a integer sequence. This conversion is performed by the `sent_to_int_seq()` method which takes in tokenized sentence like `[ 'apple' , 'orangle' ]` and maps it to a integer sequence ( consisting of tokens' indices retreived from `word_to_int` ) like `[ 1023 , 1102 ]`.

Also, we compute the maximum length of these sequences in order to perform zero padding using `pad_sequence()`

```
maxlen = max( [ len( arr ) for arr in tokenized_sentences ] )
```


In [ ]:

# Transform the given tokenized sentence to an integer sequence
# For example, [ 'apple' , 'orange' ] --> [ 1023 , 1102]
def sent_to_int_seq( sent ):
    int_seq = [ word_to_int[ token ] if token in unique_tokens else 0 for token in sent ]
    return int_seq

# Pad the given integer sequence with zeros ( from the end of the sequences )
# For example, if maxlen=5,
# [ 56 , 78 ] -> [ 56 , 78 , 0 , 0 , 0 ]
# [ 34 , 56 , 78 , 23 , 13 , 12 ] -> [ 34 , 56 , 78 , 23 , 13 ]
def pad_sequence( seq , maxlen ):
    out = np.zeros( shape=( maxlen , ) )
    out[ 0 : len( seq ) ] = seq
    return out

# Compute the max length of the tokenized sentences.
# Will be used for padding the sequences.
maxlen = max( [ len( arr ) for arr in tokenized_sentences ] )
print( f'Max Length for input sequences : {maxlen}')

# Convert tokenized_sentences to integer sequences
# Finally pad the integer sequence and store it in an array.
padded_sentences = []
for sent in tokenized_sentences:
    padded_sentences.append( pad_sequence( sent_to_int_seq( sent ) , maxlen ) )



We have cleaned the data and now we are left with integer sequences of shape `( num_samples , maxlen )` and their corresponding labels of shape `( num_samples , 1 )`.


In [ ]:

# Convert list to ndarray
x = np.array( padded_sentences )
print( x.shape )

# Reshape raw_labels from shape ( num_labels , ) to ( num_labels , 1 )
#y = raw_labels.reshape( -1 , 1) 
y = tf.keras.utils.to_categorical( raw_labels , num_classes=2 )
print( y.shape )



We split our data into training and testing datasets using `sklearn.model_selection.train_test_split()` using 20% of the data for evaluating our model.


In [ ]:

# Split the data into training and testing datasets.
train_x , test_x , train_y , test_y = sklearn.model_selection.train_test_split( x , y , test_size=0.2 )
print( train_x.shape )
print( train_y.shape )
print( test_x.shape )
print( test_y.shape )



We have completed the data preprocessing. We'll now discuss more on the training of our model.



## 2. 🤖 **Training the model**

MLP Mixer is designed originally for image classification problems, as observed in their [research paper](https://arxiv.org/abs/2105.01601). We modify the architecture and produce patches from embeddings of shape `( num_patches , embedding_dims )`. In context of textual data, `embedding_dims` could be thought as `num_channels` in an image. 

The rest of the components including MLPs and Mixer layers remain as they are.



In [ ]:

# Multilayer Perceptron with GeLU ( Gaussian Linear Units ) activation
def mlp( x , hidden_dims ):
    y = tf.keras.layers.Dense( hidden_dims )( x )
    y = tf.nn.gelu( y )
    y = tf.keras.layers.Dense( x.shape[ -1 ] )( y )
    y = tf.keras.layers.Dropout( 0.4 )( y )
    return y

# Token Mixing MLPs : Allow communication within tokens ( patches ) or, intuitively, between different parts
# of the same sequence.
def token_mixing( x , token_mixing_mlp_dims ):
    # x is a tensor of shape ( batch_size , num_patches , channels )
    x = tf.keras.layers.LayerNormalization( epsilon=1e-6 )( x )
    x = tf.keras.layers.Permute( dims=[ 2 , 1 ] )( x ) 
    # After transposition, shape of x -> ( batch_size , channels , num_patches )
    x = mlp( x , token_mixing_mlp_dims )
    return x

# Channel Mixing MLPs : Allow communication within channels ( features of embeddings )
def channel_mixing( x , channel_mixing_mlp_dims ):
    # x is a tensor of shape ( batch_size , num_patches , channels )
    x = tf.keras.layers.LayerNormalization( epsilon=1e-6 )( x )
    x = mlp( x , channel_mixing_mlp_dims )
    return x

# Mixer layer consisting of token mixing MLPs and channel mixing MLPs
# input shape -> ( batch_size , channels , num_patches )
# output shape -> ( batch_size , channels , num_patches )
def mixer( x , token_mixing_mlp_dims , channel_mixing_mlp_dims ):
    # inputs x of are of shape ( batch_size , num_patches , channels )
    # Note: "channels" is used instead of "embedding_dims"
    
    # Add token mixing MLPs
    token_mixing_out = token_mixing( x , token_mixing_mlp_dims )
    # Shape of token_mixing_out -> ( batch_size , channels , num_patches )

    token_mixing_out = tf.keras.layers.Permute( dims=[ 2 , 1 ] )( token_mixing_out )
    # Shape of transposition -> ( batch_size , num_patches , channels )
    
    #  Add skip connection
    token_mixing_out = tf.keras.layers.Add()( [ x , token_mixing_out ] )

    # Add channel mixing MLPs
    channel_mixing_out = channel_mixing( token_mixing_out , channel_mixing_mlp_dims )
    # Shape of channel_mixing_out -> ( batch_size , num_patches , channels )
    
    # Add skip connection
    channel_mixing_out = tf.keras.layers.Add()( [ channel_mixing_out , token_mixing_out ] )
    # Shape of channel_mixing_out -> ( batch_size , num_patches , channels )

    return channel_mixing_out



Compile the model with `SparseCategoricalFocalLoss` and `Adam` optimizer. We'll use the class weights, which we computed earlier, in the `model.fit()` method.


In [ ]:

# These hyperparameters were searched with KerasTuner
embedding_dims = 64
token_mixing_mlp_dims = 32
channel_mixing_mlp_dims = 64
patch_size = 5
num_mixer_layers = 8
learning_rate = 5e-3

num_classes = 2
seq_input_shape = ( maxlen , )
    
# Model input layer
inputs = tf.keras.layers.Input( shape=seq_input_shape )

# Embedding layer which converts int sequences into dense vectors
embedding = tf.keras.layers.Embedding( input_dim=vocab_size + 1 , output_dim=embedding_dims , input_length=maxlen )( inputs )
    
# Conv1D layer to produce patches from given sequences. 
patches = tf.keras.layers.Conv1D( embedding_dims ,
                                 kernel_size=patch_size ,
                                 strides=patch_size ,
                                 use_bias=False , 
                                 trainable=False )( embedding )
    
x = patches
for _ in range( num_mixer_layers ):
    x = mixer( x , token_mixing_mlp_dims , channel_mixing_mlp_dims )
        
x = tf.keras.layers.LayerNormalization( epsilon=1e-6 )( x )
x = tf.keras.layers.GlobalAveragePooling1D()( x )
outputs = tf.keras.layers.Dense( num_classes , activation='softmax' )( x )

model = tf.keras.models.Model( inputs , outputs )
model.summary()


In [ ]:

# Batch size and epochs
batch_size = 32
num_epochs = 5

# Compile the model and start the training
model.compile(
    loss='categorical_crossentropy' , 
    optimizer=tf.keras.optimizers.Adam( learning_rate ) ,
    metrics=[ 'accuracy' ]
)

model.fit(train_x ,
          train_y ,
          batch_size=batch_size ,
          validation_data=( test_x , test_y ) ,
          epochs=num_epochs , 
          class_weight=class_weights ,
        )



## 3. 🦾 **Evaluating the model**

![Evaluation Metrics](https://static.packt-cdn.com/products/9781785282287/graphics/B04223_10_02.jpg)

> Image Source: [Computing precision, recall, and F1-score - Packt](https://subscription.packtpub.com/book/big_data_and_business_intelligence/9781785282287/10/ch10lvl1sec133/computing-precision-recall-and-f1-score)

After training our model, we'll like to evaluate it using our test data. Using `sklearn.metrics.classification_report` we examine the precision, recall and f1 scores for the two classes, `disaster` and `not disaster`.


In [ ]:

# Fetch model predictions for test_x
pred_y = model.predict( test_x )

# Print the classification report
report = sklearn.metrics.classification_report( np.argmax( test_y , axis=1 ) , np.argmax( pred_y , axis=1 ) , target_names=[ 'not disaster' , 'disaster' ] )
print( report )



Alongside, we plot the confusion matrix


In [ ]:

# Plot the confusion matrix
conf_matrix = sklearn.metrics.confusion_matrix( np.argmax( test_y , axis=1 ) ,np.argmax( pred_y , axis=1 ) )
disp = sklearn.metrics.ConfusionMatrixDisplay( conf_matrix , display_labels=[ 'not disaster' , 'disaster' ] )
disp.plot() 



## 4. ✍️ **Submitting the results**

For submitting our predictions to the competition, we need to write them in a CSV file called `submission.csv`. We parse the tweets from `test.csv`, clean them and generate perdictions for them. These predictions are stored with their corresponding ids in `submission.csv`.


In [ ]:

# Read the test.csv file
test_df = pd.read_csv( '../input/nlp-getting-started/test.csv' , usecols=[ 'text' , 'id' ] )

# Clean, tokenize, pad the sentences from test_df
test_inputs = []
for text in test_df[ 'text' ].values:
    tokens = process_sent( text )
    out = sent_to_int_seq( tokens )
    out = pad_sequence( out , maxlen )
    test_inputs.append( out )
    
# Fetch predictions for test_inputs
test_inputs = np.array( test_inputs )
predicted_labels = np.argmax( model.predict( test_inputs ) , axis=1 )
    
ids = test_df[ 'id' ].values

# Create the submission.csv file from ids and predicted_labels
submission_csv = { 'id' : ids , 'target' : predicted_labels }
submission_csv = pd.DataFrame.from_dict( submission_csv )
submission_csv.to_csv( 'submission.csv' , index=False )
